## Data Cleaning Pipeline - NF-UQ-NIDS-v2 Dataset

Cleaning and balancing pipeline for NF-UQ-NIDS-v2

**Steps:**
1. Convert large CSV to Parquet in chunks
2. Analyze label distribution
3. Balance dataset (downsample Benign + large attack classes)
4. Export balanced dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
import os, glob, gc, time, warnings
warnings.filterwarnings('ignore')



## Convert Large CSV to Parquet in Chunks
- Optimize memory usage (downcast data types)


In [2]:
RAW_CSV     = "/home/huyho/earth_predict_env/dataset/NF-UQ-NIDS/NF-UQ-NIDS-v2.csv"
PARQUET_DIR = "/home/huyho/earth_predict_env/dataset/NF-UQ-NIDS/parquet_data"
os.makedirs(PARQUET_DIR, exist_ok=True)

DROP_COLS  = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']
LABEL_COL  = 'Attack'
BINARY_COL = 'Label'
CHUNK_SIZE = 500_000

def reduce_memory(df):
    for col in df.columns:
        if col in (LABEL_COL, BINARY_COL):
            continue
        col_type = df[col].dtype
        if col_type == object:
            continue
        c_min, c_max = df[col].min(), df[col].max()
        if str(col_type)[:3] == 'int':
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        elif str(col_type)[:5] == 'float':
            df[col] = df[col].astype(np.float32)
    return df

def fix_numeric_cols(df):
    skip = {LABEL_COL, BINARY_COL}
    for col in df.columns:
        if col not in skip and df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

existing = sorted(glob.glob(os.path.join(PARQUET_DIR, "chunk_*.parquet")))

start = time.time()
chunk_idx, total_rows = 0, 0
for chunk in pd.read_csv(RAW_CSV, chunksize=CHUNK_SIZE, low_memory=False):
    chunk_idx += 1
    cols_to_drop = [c for c in DROP_COLS if c in chunk.columns]
    chunk = chunk.drop(columns=cols_to_drop)
    chunk = fix_numeric_cols(chunk)
    chunk = reduce_memory(chunk)
    if chunk[BINARY_COL].dtype == object:
        chunk = chunk[chunk[BINARY_COL] != str(BINARY_COL)]
    chunk = chunk.dropna(subset=[LABEL_COL, BINARY_COL])
    out = os.path.join(PARQUET_DIR, f"chunk_{chunk_idx:04d}.parquet")
    chunk.to_parquet(out, index=False)
    total_rows += len(chunk)
    del chunk; gc.collect()
elapsed = time.time() - start


## Choose Labels and Sample Sizes

Edit `SAMPLING_CONFIG` to select attack types and target sample counts.  
Set `None` to keep all rows for that label.


In [ ]:
# Format: 'LabelName': sample_count  (set None to keep all rows for that label)
SAMPLING_CONFIG = {
    'Benign'       : 400000,
    'DoS'          : 300000,
    'DDoS'         : 300000,
    'Brute Force'  : 300000,
    'Infilteration': 300000,
}

## Balance Dataset and Export Balanced File

**Balancing strategy:**
- Downsample **Benign** 
- Downsample large attack classes
- Keep small attack classes unchanged 


In [5]:
SELECTED_LABELS = list(SAMPLING_CONFIG.keys())

PARQUET_DIR   = "/home/huyho/earth_predict_env/dataset/NF-UQ-NIDS/parquet_data"
parquet_files = sorted(glob.glob(os.path.join(PARQUET_DIR, "chunk_*.parquet")))

df_dask    = dd.read_parquet(parquet_files)
label_dist = df_dask['Attack'].value_counts().compute().sort_values(ascending=False)
total_rows = label_dist.sum()
label_pct  = (label_dist / total_rows * 100).round(2)

sampled_parts = []
for label, target_n in SAMPLING_CONFIG.items():
    total = df_dask[df_dask['Attack'] == label].shape[0].compute()
    if total == 0:
        continue
    if target_n is None or total <= target_n:
        df_part = df_dask[df_dask['Attack'] == label].compute()
    else:
        frac = target_n / total
        df_part = df_dask[df_dask['Attack'] == label].sample(frac=frac, random_state=42).compute()
        if len(df_part) > target_n:
            df_part = df_part.sample(n=target_n, random_state=42)
    sampled_parts.append(df_part)
    del df_part; gc.collect()

df_balanced = pd.concat(sampled_parts, ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
df_balanced['Label'] = df_balanced['Attack']
df_balanced = df_balanced.drop(columns=['Attack'])
del sampled_parts; gc.collect()

label_counts  = df_balanced['Label'].value_counts()
label_percent = (label_counts / label_counts.sum() * 100).sort_values(ascending=False)
label_counts  = label_counts[label_percent.index]

for lbl in label_percent.index:
    print(f"{lbl:<30} {label_counts[lbl]:>15,} {label_percent[lbl]:>10.2f}")


Benign                                 400,000      32.25
DDoS                                   300,000      24.19
DoS                                    299,999      24.19
Brute Force                            123,982      10.00
Infilteration                          116,361       9.38


In [ ]:
DATASET_OUTPUT_DIR = "/home/huyho/earth_predict_env/dataset/NF-UQ-NIDS"
os.makedirs(DATASET_OUTPUT_DIR, exist_ok=True)

balanced_output = os.path.join(DATASET_OUTPUT_DIR, "nf_uq_balanced_dataset.parquet")
df_balanced.to_parquet(balanced_output, index=False)
